In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Dữ liệu/student_data/inventory.csv")
df.head(5)

,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,fill_rate,stockout_flag,overstock_flag,reorder_flag,sell_through_rate,product_name,category,segment,year,month
0,2022-10-31,1,3,1,1,2,90.0,0.9333,1,0,0,0.2500,DragonWear MA-01,Casual,All-weather,2022,10
1,2022-11-30,1,3,1,1,1,90.0,0.9667,1,0,0,0.2500,DragonWear MA-01,Casual,All-weather,2022,11
2,2022-12-31,1,3,1,1,1,90.0,0.9667,1,0,0,0.2500,DragonWear MA-01,Casual,All-weather,2022,12
3,2016-04-30,3,35,13,11,2,95.5,0.9333,1,1,0,0.2391,DragonWear MA-03,Casual,All-weather,2016,4
4,2016-05-31,3,36,11,10,1,108.0,0.9667,1,1,0,0.2174,DragonWear MA-03,Casual,All-weather,2016,5


In [4]:
# ============================================================
# 1. IMPORT THƯ VIỆN
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 2. ĐỌC DỮ LIỆU GỐC
# ============================================================

# Nếu bạn đã lưu file trên Google Drive,
# hãy thay đường dẫn bên dưới bằng đường dẫn thực tế.

file_path = "/content/drive/MyDrive/Colab Notebooks/Dữ liệu/student_data/inventory.csv"

df = pd.read_csv(file_path)

print("Đọc dữ liệu thành công!")
print("Kích thước dữ liệu:", df.shape)

display(df.head(5))


# ============================================================
# 3. KIỂM TRA DỮ LIỆU GỐC
# ============================================================

print("========== THÔNG TIN DỮ LIỆU GỐC ==========")

print("\n1. Kích thước dữ liệu:")
print(df.shape)

print("\n2. Thông tin các cột:")
df.info()

print("\n3. Số lượng giá trị thiếu:")
print(df.isnull().sum())

print("\n4. Số dòng bị trùng hoàn toàn:")
print(df.duplicated().sum())

print("\n5. Tên các cột:")
print(df.columns.tolist())


# ============================================================
# 4. SAO CHÉP DỮ LIỆU
# ============================================================

df_clean = df.copy()


# ============================================================
# 5. CHUẨN HÓA TÊN CỘT
# ============================================================

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
)

print("\nTên cột sau khi chuẩn hóa:")
print(df_clean.columns.tolist())


# ============================================================
# 6. CHUYỂN ĐỔI KIỂU DỮ LIỆU
# ============================================================

# Chuyển snapshot_date sang kiểu ngày tháng
df_clean["snapshot_date"] = pd.to_datetime(
    df_clean["snapshot_date"],
    errors="coerce"
)

# Các cột số nguyên
integer_columns = [
    "product_id",
    "stock_on_hand",
    "units_received",
    "units_sold",
    "stockout_days",
    "stockout_flag",
    "overstock_flag",
    "reorder_flag"
]

for column in integer_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

# Các cột số thực
float_columns = [
    "days_of_supply",
    "fill_rate",
    "sell_through_rate"
]

for column in float_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )


# ============================================================
# 7. KIỂM TRA GIÁ TRỊ THIẾU SAU KHI CHUYỂN ĐỔI
# ============================================================

print("\n========== GIÁ TRỊ THIẾU ==========")
print(df_clean.isnull().sum())


# ============================================================
# 8. XÓA CÁC DÒNG THIẾU DỮ LIỆU QUAN TRỌNG
# ============================================================

required_columns = [
    "snapshot_date",
    "product_id",
    "stock_on_hand",
    "units_received",
    "units_sold",
    "stockout_days",
    "days_of_supply",
    "stockout_flag",
    "overstock_flag",
    "reorder_flag",
    "sell_through_rate"
]

df_clean = df_clean.dropna(
    subset=required_columns
)


# ============================================================
# 9. XÓA CÁC DÒNG TRÙNG LẶP HOÀN TOÀN
# ============================================================

df_clean = df_clean.drop_duplicates()


# ============================================================
# 10. KIỂM TRA KHÓA CHÍNH
# ============================================================

# Theo thiết kế bảng INVENTORY:
# Một sản phẩm có thể xuất hiện nhiều ngày khác nhau.
# Vì vậy, khóa chính nên là:
# (snapshot_date, product_id)

duplicate_keys = df_clean.duplicated(
    subset=["snapshot_date", "product_id"]
)

print("\nSố dòng bị trùng khóa chính:")
print(duplicate_keys.sum())

# Loại bỏ bản ghi trùng khóa chính nếu có
df_clean = df_clean.drop_duplicates(
    subset=["snapshot_date", "product_id"],
    keep="first"
)


# ============================================================
# 11. KIỂM TRA DỮ LIỆU KHÔNG HỢP LỆ
# ============================================================

# Các số lượng tồn kho và số lượng sản phẩm
# không được nhỏ hơn 0

df_clean = df_clean[
    (df_clean["stock_on_hand"] >= 0) &
    (df_clean["units_received"] >= 0) &
    (df_clean["units_sold"] >= 0) &
    (df_clean["stockout_days"] >= 0)
]


# Số ngày hết hàng không được vượt quá 31 ngày
df_clean = df_clean[
    df_clean["stockout_days"] <= 31
]


# days_of_supply không được âm
df_clean = df_clean[
    df_clean["days_of_supply"] >= 0
]


# sell_through_rate phải nằm trong khoảng 0 đến 1
df_clean = df_clean[
    df_clean["sell_through_rate"].between(0, 1)
]


# Các cờ chỉ được nhận giá trị 0 hoặc 1
df_clean = df_clean[
    df_clean["stockout_flag"].isin([0, 1]) &
    df_clean["overstock_flag"].isin([0, 1]) &
    df_clean["reorder_flag"].isin([0, 1])
]


# product_id phải là số nguyên dương
df_clean = df_clean[
    df_clean["product_id"] > 0
]


# ============================================================
# 12. CHUYỂN CÁC CỘT SỐ NGUYÊN VỀ KIỂU INT
# ============================================================

for column in integer_columns:
    df_clean[column] = df_clean[column].astype(int)


# ============================================================
# 13. SẮP XẾP DỮ LIỆU
# ============================================================

df_clean = df_clean.sort_values(
    by=["snapshot_date", "product_id"]
).reset_index(drop=True)


# ============================================================
# 14. CHỌN CÁC CỘT ĐÚNG VỚI BẢNG INVENTORY
# ============================================================

# Theo sơ đồ CSDL của bạn, bảng INVENTORY gồm:

inventory_columns = [
    "snapshot_date",
    "product_id",
    "stock_on_hand",
    "units_received",
    "units_sold",
    "stockout_days",
    "days_of_supply",
    "stockout_flag",
    "overstock_flag",
    "reorder_flag",
    "sell_through_rate"
]

Inventory = df_clean[inventory_columns].copy()


# ============================================================
# 15. ĐỊNH DẠNG CỘT SNAPSHOT_DATE
# ============================================================

Inventory["snapshot_date"] = (
    Inventory["snapshot_date"].dt.date
)


# ============================================================
# 16. KIỂM TRA KẾT QUẢ SAU KHI LÀM SẠCH
# ============================================================

print("\n========== KẾT QUẢ SAU KHI LÀM SẠCH ==========")

print("\n1. Kích thước dữ liệu:")
print(Inventory.shape)

print("\n2. Số dòng dữ liệu:")
print(len(Inventory))

print("\n3. Số giá trị thiếu:")
print(Inventory.isnull().sum())

print("\n4. Số dòng trùng hoàn toàn:")
print(Inventory.duplicated().sum())

print("\n5. Số khóa chính bị trùng:")
print(
    Inventory.duplicated(
        subset=["snapshot_date", "product_id"]
    ).sum()
)

print("\n6. Ngày bắt đầu:")
print(Inventory["snapshot_date"].min())

print("\n7. Ngày kết thúc:")
print(Inventory["snapshot_date"].max())

print("\n8. Số sản phẩm:")
print(Inventory["product_id"].nunique())

print("\n9. Tên các cột:")
print(Inventory.columns.tolist())

print("\n10. Năm dòng dữ liệu sau khi làm sạch:")
display(Inventory.head())


# ============================================================
# 17. KIỂM TRA TÍNH HỢP LỆ
# ============================================================

# Không có giá trị thiếu
assert Inventory.notna().all().all()

# Khóa chính kết hợp không trùng
assert Inventory.duplicated(
    subset=["snapshot_date", "product_id"]
).sum() == 0

# Số lượng không âm
assert Inventory["stock_on_hand"].ge(0).all()
assert Inventory["units_received"].ge(0).all()
assert Inventory["units_sold"].ge(0).all()
assert Inventory["stockout_days"].ge(0).all()

# days_of_supply không âm
assert Inventory["days_of_supply"].ge(0).all()

# Tỷ lệ bán hàng hợp lệ
assert Inventory["sell_through_rate"].between(0, 1).all()

# Các cờ chỉ nhận 0 hoặc 1
assert Inventory["stockout_flag"].isin([0, 1]).all()
assert Inventory["overstock_flag"].isin([0, 1]).all()
assert Inventory["reorder_flag"].isin([0, 1]).all()

print("\nDữ liệu INVENTORY đã vượt qua các kiểm tra cơ bản!")


# ============================================================
# 18. THỐNG KÊ MÔ TẢ DỮ LIỆU
# ============================================================

print("\n========== THỐNG KÊ MÔ TẢ ==========")

display(Inventory.describe())


# ============================================================
# 19. LƯU DỮ LIỆU ĐÃ LÀM SẠCH THÀNH FILE CSV
# ============================================================

output_path = "/content/Inventory_clean.csv"

Inventory.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu dữ liệu sạch tại:")
print(output_path)


# ============================================================
# 20. HIỂN THỊ DỮ LIỆU TRONG BẢNG INVENTORY
# ============================================================

print("\n========== BẢNG INVENTORY ==========")

display(Inventory)

Đọc dữ liệu thành công!
Kích thước dữ liệu: (60247, 17)


,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,fill_rate,stockout_flag,overstock_flag,reorder_flag,sell_through_rate,product_name,category,segment,year,month
0,2022-10-31,1,3,1,1,2,90.0,0.9333,1,0,0,0.2500,DragonWear MA-01,Casual,All-weather,2022,10
1,2022-11-30,1,3,1,1,1,90.0,0.9667,1,0,0,0.2500,DragonWear MA-01,Casual,All-weather,2022,11
2,2022-12-31,1,3,1,1,1,90.0,0.9667,1,0,0,0.2500,DragonWear MA-01,Casual,All-weather,2022,12
3,2016-04-30,3,35,13,11,2,95.5,0.9333,1,1,0,0.2391,DragonWear MA-03,Casual,All-weather,2016,4
4,2016-05-31,3,36,11,10,1,108.0,0.9667,1,1,0,0.2174,DragonWear MA-03,Casual,All-weather,2016,5


========== THÔNG TIN DỮ LIỆU GỐC ==========

1. Kích thước dữ liệu:
(60247, 17)

2. Thông tin các cột:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60247 entries, 0 to 60246
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   snapshot_date      60247 non-null  object 
 1   product_id         60247 non-null  int64  
 2   stock_on_hand      60247 non-null  int64  
 3   units_received     60247 non-null  int64  
 4   units_sold         60247 non-null  int64  
 5   stockout_days      60247 non-null  int64  
 6   days_of_supply     60247 non-null  float64
 7   fill_rate          60247 non-null  float64
 8   stockout_flag      60247 non-null  int64  
 9   overstock_flag     60247 non-null  int64  
 10  reorder_flag       60247 non-null  int64  
 11  sell_through_rate  60247 non-null  float64
 12  product_name       60247 non-null  object 
 13  category           60247 non-null  object 
 14  segment        

,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,stockout_flag,overstock_flag,reorder_flag,sell_through_rate
0,2012-07-31,4,22,8,7,2,94.3,1,1,0,0.2414
1,2012-07-31,31,18,6,6,2,90.0,1,0,0,0.2500
2,2012-07-31,36,24,8,8,0,90.0,0,0,0,0.2500
3,2012-07-31,86,6,2,2,0,90.0,0,0,0,0.2500
4,2012-07-31,93,41,15,13,2,94.6,1,1,0,0.2407



Dữ liệu INVENTORY đã vượt qua các kiểm tra cơ bản!

========== THỐNG KÊ MÔ TẢ ==========


,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,stockout_flag,overstock_flag,reorder_flag,sell_through_rate
count,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.000000,60247.0,60247.000000
mean,1311.408468,189.298455,18.046807,15.417764,1.160639,912.677576,0.673411,0.762561,0.0,0.152275
std,673.051769,316.976124,34.080228,28.404379,1.624490,2587.624108,0.468969,0.425517,0.0,0.139291
min,1.000000,3.000000,1.000000,1.000000,0.000000,5.200000,0.000000,0.000000,0.0,0.000400
25%,760.000000,15.000000,2.000000,2.000000,0.000000,96.000000,0.000000,1.000000,0.0,0.042100
50%,1223.000000,62.000000,6.000000,6.000000,1.000000,240.000000,1.000000,1.000000,0.0,0.111100
75%,1942.000000,210.000000,19.000000,16.000000,2.000000,683.100000,1.000000,1.000000,0.0,0.238100
max,2412.000000,2673.000000,817.000000,670.000000,28.000000,68100.000000,1.000000,1.000000,0.0,0.853100



Đã lưu dữ liệu sạch tại:
/content/Inventory_clean.csv

========== BẢNG INVENTORY ==========


,snapshot_date,product_id,stock_on_hand,units_received,units_sold,stockout_days,days_of_supply,stockout_flag,overstock_flag,reorder_flag,sell_through_rate
0,2012-07-31,4,22,8,7,2,94.3,1,1,0,0.2414
1,2012-07-31,31,18,6,6,2,90.0,1,0,0,0.2500
2,2012-07-31,36,24,8,8,0,90.0,0,0,0,0.2500
3,2012-07-31,86,6,2,2,0,90.0,0,0,0,0.2500
4,2012-07-31,93,41,15,13,2,94.6,1,1,0,0.2407
...,...,...,...,...,...,...,...,...,...,...,...
60242,2022-12-31,2396,1254,5,4,0,9405.0,0,1,0,0.0032
60243,2022-12-31,2402,172,1,1,2,5160.0,1,1,0,0.0058
60244,2022-12-31,2403,87,21,18,2,145.0,1,1,0,0.1714
60245,2022-12-31,2405,63,1,1,0,1890.0,0,1,0,0.0156
